# Global Daily SST Analysis: Identifying Marine Extremes with `MarEx-Detect`

This notebook detects marine heatwaves in 20 years (1995-2014) of daily sea-surface temperature from the ICON-ESM-ER
`hist-1950` simulation of the EERIE project, on the model's native R02B09 ocean mesh: 14.9 M triangular cells at about
5 km spacing. The method and parameters are those of Part 1 of the gridded example (`shifting_baseline` with a 15-year
baseline, `seasonal_percentile` at the 95th percentile), so the two can be compared directly. The baseline consumes the
first 15 years as spin-up, leaving five years of output (2010-2014) for the tracking example,
`02_id_track_events.ipynb`.

Everything runs on a single node with 64 cores and 128 GB. Each stage prints its wall time and peak memory as it
finishes, and the table at the end collects them. Unstructured input is described under
[Preparing Your Data](https://marex.readthedocs.io/en/latest/guide/dimensions_and_time.html#gridded-and-unstructured-inputs), and the detection methods under
[Choosing an Anomaly Method](https://marex.readthedocs.io/en/latest/guide/anomalies.html#choosing-a-method).

In [ ]:
from getpass import getuser
from pathlib import Path

import intake
import xarray as xr

import marEx
import marEx.helper as hpc

In [ ]:
scratch_dir = Path("/scratch") / getuser()[0] / getuser()  # Lustre scratch
out_dir = scratch_dir / "marEx" / "examples" / "unstructured"

# ICON R02B09 ocean grid on Levante: neighbour indices, cell areas and cell centres
grid_file = "/pool/data/ICON/grids/public/mpim/0016/icon_grid_0016_R02B09_O.nc"

period = slice("1995-01-01", "2014-12-31")  # 20 years of input
window_years = 15  # shifting-baseline length in years. The output starts this many years after `period`

# One node: 4 workers x 16 threads x 28 GB = 64 cores, 112 GB for the workers, the rest of 128 GB for the client
n_workers, threads_per_worker, memory_limit = 4, 16, "28GB"

In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)

# memory_limit is passed explicitly because on an HPC node Dask sizes workers from the whole node's memory,
# not from the job's allocation, and without it the workers can claim more than the job was granted.
client = hpc.start_local_cluster(
    n_workers=n_workers,
    threads_per_worker=threads_per_worker,
    memory_limit=memory_limit,
    scratch_dir=scratch_dir / "marEx" / "dask",
)

# Records wall time, peak memory, spill and worker restarts for each `monitor.stage(...)` block below
monitor = hpc.ResourceMonitor(client)

### Input Chunking

This is the setting that decides whether the notebook runs at all. The climatology and percentile reductions are global
in time and independent in space, so marEx works on narrow-in-space, full-in-time slabs, and one slab's working set is
`n_time × (cells in one input spatial chunk) × 4 B`. It depends on the spatial chunk width, not the time chunk.

Leaving `ncells` whole makes that 7305 × 14.9 M × 4 B ≈ 435 GB, and the internal rechunk becomes an all-to-all transpose
of the whole array, which thrashes rather than failing. Chunking `ncells` to 25,000 bounds a slab at ≈ 0.73 GB. The
time chunk must be at least `smooth_days` (21), because the 21-day centred rolling mean is applied per chunk. The
tracker needs the opposite layout, with space whole (see `02_id_track_events.ipynb`). More under
[Memory Management and Chunking](https://marex.readthedocs.io/en/latest/guide/performance.html#chunking).

In [ ]:
cat = intake.open_catalog("https://raw.githubusercontent.com/eerie-project/intake_catalogues/main/eerie.yaml")
icon = cat["dkrz.disk.model-output"]["icon-esm-er"]["hist-1950"]["v20240618"]["ocean"]["native"]

sst = (
    icon["2d_daily_mean"](chunks={})
    .to_dask()
    .to.isel(depth=0)  # surface level of potential temperature
    .drop_vars({"cell_sea_land_mask"})
    .sel(time=period)
    .chunk({"time": 21, "ncells": 25_000})
)
sst

### Grid Connectivity

The EERIE catalogue has no grid entry for `hist-1950`, so `grid_file` points directly at the ICON R02B09 ocean grid.

marEx is not tied to ICON, but it does assume a triangular mesh. A substitute `grid_file` must provide:

- `neighbor_cell_index` with dimensions `(nv, ncells)` and shape `(3, ncells)`: exactly three neighbours per cell, which
  the dilation operator assumes
- 1-based neighbour indices: marEx subtracts 1, and any value that then falls below 0 means "no neighbour" (a mesh
  boundary)
- `cell_area`, and `clat`/`clon` cell-centre coordinates (renamed to `lat`/`lon` below)

Triangular meshes such as ICON and FESOM satisfy this. A hexagonal mesh (e.g. MPAS) would need changes to the dilation
matrix.

In [ ]:
grid2d = xr.open_dataset(grid_file, chunks={}).rename({"cell": "ncells"})
neighbours = grid2d.neighbor_cell_index.rename({"clat": "lat", "clon": "lon"})
areas = grid2d.cell_area.rename({"clat": "lat", "clon": "lon"})

## Pre-Process with Shifting-Baseline & Hobday et al. (2016) Definition for Extremes

In [ ]:
with monitor.stage("Detect: shifting baseline + Hobday"):
    extremes_ds = marEx.preprocess_data(
        sst,
        method_anomaly="shifting_baseline",  # anomaly from a backward-looking climatology of the previous `window_years`
        method_extreme="seasonal_percentile",  # day-of-year threshold (Hobday et al., 2016)
        threshold_percentile=95,
        window_years=window_years,
        smooth_days=21,  # smoothing applied to the climatology (days)
        window_days=11,  # samples from an 11-day window around each day of year enter its percentile
        dask_chunks={"time": 5},  # output chunking, read by the tracker in 02 without a rechunk
        neighbours=neighbours,  # attached to the output for the tracker
        cell_areas=areas,  # m², attached to the output for the tracker
        dimensions={"time": "time", "x": "ncells"},  # without a "y" dimension marEx treats the data as unstructured
        coordinates={"time": "time", "x": "lon", "y": "lat"},
    )
    extremes_ds.to_zarr(out_dir / "extremes_shifting_hobday.zarr", mode="w")
extremes_ds

## Wall Time and Memory

One row per stage, measured while the notebook ran on the cluster configured at the top. Peak worker memory is the
summed process memory the workers report to the scheduler, which is the quantity Dask compares against `memory_limit`.
The client (this kernel) is counted separately.

In [ ]:
monitor.summary()

In [ ]:
client.close()